In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gspread
from google.oauth2.service_account import Credentials
import statsmodels.api as sm
import re

In [ ]:
SHEET_ID = "1GOYwVG_bB-VL0tEzW7UisO7K8akTQYowq7SKgdH1QCk"
WORKSHEET_GID = 1713119709
CREDENTIALS_FILE = os.path.join(os.path.dirname(os.path.abspath("healthTrackerReport.ipynb")), "stalwart-micron-440802-v6-6f824c7a3859.json")

creds = Credentials.from_service_account_file(
    CREDENTIALS_FILE,
    scopes=["https://www.googleapis.com/auth/spreadsheets.readonly"]
)
gc = gspread.authorize(creds)

In [ ]:
spreadsheet = gc.open_by_key(SHEET_ID)
worksheet = spreadsheet.get_worksheet_by_id(WORKSHEET_GID)
df = pd.DataFrame(worksheet.get_all_records())

df.columns

In [ ]:
df['Date of Entry']

In [ ]:
df['Date of Entry'] = pd.to_datetime(df['Date of Entry'], errors='coerce')

cutoff = pd.Timestamp.now().normalize() - pd.DateOffset(months=1)

df_filtered = df[df['Date of Entry'] >= cutoff]

In [ ]:
ax = df_filtered['Activities/Environment [Did I take NSAID?]'] \
        .value_counts() \
        .plot(kind='bar')

# add counts on top of bars
for container in ax.containers:
    ax.bar_label(container)

plt.tight_layout()

In [ ]:
ax = df_filtered['Activities/Environment [Did I take RizoTriptan?]'] \
        .value_counts() \
        .plot(kind='bar')

# add counts on top of bars
for container in ax.containers:
    ax.bar_label(container)

plt.tight_layout()

In [ ]:
df_filtered[['Date of Entry', 'Did I take any other drugs?']].tail(5)

In [ ]:
df_filtered['Timestamp'].head()

In [ ]:
helped = df['What were the things that helped today?'] \
    .dropna() \
    .replace('', pd.NA) \
    .dropna()

print('\n'.join(helped))

In [ ]:
triggered = df['Triggers today and other comments'] \
    .dropna() \
    .replace('', pd.NA) \
    .dropna()

print('\n'.join(triggered))

In [ ]:
new_symptoms = df['Any new symptoms?'] \
    .dropna() \
    .replace('', pd.NA) \
    .dropna()

print('\n'.join(new_symptoms))

# Regressions

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def vif_table(X_df):
    X_const = sm.add_constant(X_df)
    return pd.DataFrame({
        'Variable': X_df.columns,
        'VIF': [variance_inflation_factor(X_const.values, i + 1)
                for i in range(X_df.shape[1])]
    }).sort_values('VIF', ascending=False).reset_index(drop=True)

In [ ]:
TEXT_COLS = [
    'Timestamp', 'Date of Entry',
    'What were the things that helped today?',
    'Any new symptoms?', 'Did I take any other drugs?',
    'Triggers today and other comments',
]

reg_df = df.copy()

# Symptom severity: "3 (Moderate)" → 3.0
symptom_cols = [c for c in reg_df.columns if c.startswith('Symptom Severity Rating')]
for col in symptom_cols:
    reg_df[col] = reg_df[col].astype(str).str.extract(r'^(\d)')[0].astype(float)

# Yes/No activities → 1/0
activity_cols = [c for c in reg_df.columns if c.startswith('Activities/Environment')]
for col in activity_cols:
    reg_df[col] = reg_df[col].map({'Yes': 1, 'No': 0})

reg_df = reg_df.drop(columns=[c for c in TEXT_COLS if c in reg_df.columns])
reg_df = reg_df.dropna()

# Shorten column names so statsmodels summary table isn't truncated
def shorten(col):
    m = re.search(r'\[(.+?)\]', col)
    return m.group(1).strip() if m else col

reg_df = reg_df.rename(columns=shorten)

print(f"{len(reg_df)} complete rows available for regression")
print(reg_df.columns.tolist())

## Neck Pain

In [ ]:
NECK_EXCLUDE = [
    'Other Numbness (explain below)',
    'Did I take Nurtec?',
    'Did I take RizoTriptan?',
    'Did I take an antihistamine?',
    'Did I take Flovent?',
    'Did I take Albuterol?',
    'Did I take Nasonex?',
    'Did I use Azelastine?',
]

X_neck = reg_df.drop(columns=['Neck Pain'] + NECK_EXCLUDE)
vif_table(X_neck)

In [ ]:
y_neck = reg_df['Neck Pain']
X = sm.add_constant(X_neck)

# First pass: keep predictors with p < 0.3
first_pass = sm.OLS(y_neck, X).fit()
keep = first_pass.pvalues[first_pass.pvalues < 0.3].index.tolist()
# print(first_pass.summary())

# Second pass: rerun with selected predictors only
model_neck = sm.OLS(y_neck, X[keep]).fit()
print(model_neck.summary())

## Headache

In [ ]:
X_head = reg_df.drop(columns=['Headache'])
vif_table(X_head)

In [ ]:
y_head = reg_df['Headache']
X = sm.add_constant(X_head)

# First pass: keep predictors with p < 0.3
first_pass = sm.OLS(y_head, X).fit()
keep = first_pass.pvalues[first_pass.pvalues < 0.3].index.tolist()

# Second pass: rerun with selected predictors only
model_head = sm.OLS(y_head, X[keep]).fit()
print(model_head.summary())